# Bài tập Perceptron – Phân khúc nhà

Sử dụng `Housing.csv` để tạo nhãn phân khúc, huấn luyện **Perceptron**, đánh giá mô hình và dự đoán nhà mới.

Quy trình: đọc dữ liệu → tạo `price_per_area` → tạo nhãn → train/test → tiền xử lý → Perceptron → đánh giá → xuất CSV → dự đoán.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [4]:
# Housing.csv đặt cùng thư mục với notebook
BASE_DIR = Path.cwd()
CSV_FILE = BASE_DIR / "Housing.csv"

print("Đường dẫn CSV:", CSV_FILE)

if not CSV_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Housing.csv tại: {CSV_FILE}\n"
        "Hãy đặt Housing.csv cùng thư mục notebook."
    )

df = pd.read_csv(CSV_FILE)
print("Đọc dữ liệu thành công!")
print("Kích thước:", df.shape)
display(df.head())


Đường dẫn CSV: d:\AI\Documents\Downloads\lac-hong-neural-network\Housing.csv
Đọc dữ liệu thành công!
Kích thước: (545, 13)


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [5]:
# Tạo nhãn phân khúc
# Housing.csv không ghi rõ đơn vị của price và area.
# Với dữ liệu hiện tại dùng ngưỡng 850 để tạo 2 lớp.
THRESHOLD = 850.0

df["price_per_area"] = df["price"] / df["area"]

df["segment"] = (df["price_per_area"] >= THRESHOLD).astype(int)

CLASS_NAMES = {
    0: "Phổ thông",
    1: "Giá/mật độ giá cao"
}

df["phan_khuc"] = df["segment"].map(CLASS_NAMES)

print("Ngưỡng price/area:", THRESHOLD)
print("\nPhân bố:")
print(df["phan_khuc"].value_counts())


Ngưỡng price/area: 850.0

Phân bố:
phan_khuc
Giá/mật độ giá cao    345
Phổ thông             200
Name: count, dtype: int64


In [6]:
# Feature đầu vào
# Không dùng price và price_per_area vì chúng trực tiếp tạo nhãn.
NUMERIC_FEATURES = [
    "area", "bedrooms", "bathrooms", "stories", "parking"
]

CATEGORICAL_FEATURES = [
    "mainroad", "guestroom", "basement", "hotwaterheating",
    "airconditioning", "prefarea", "furnishingstatus"
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURE_COLUMNS].copy()
y = df["segment"].copy()

print("Feature:")
for f in FEATURE_COLUMNS:
    print("-", f)


Feature:
- area
- bedrooms
- bathrooms
- stories
- parking
- mainroad
- guestroom
- basement
- hotwaterheating
- airconditioning
- prefarea
- furnishingstatus


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (408, 12)
Test : (137, 12)


In [8]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         CATEGORICAL_FEATURES)
    ]
)

model = Pipeline([
    ("preprocessor", preprocess),
    ("perceptron", Perceptron(
        max_iter=1000,
        tol=1e-3,
        eta0=0.1,
        random_state=42,
        shuffle=True
    ))
])

model.fit(X_train, y_train)
print("Huấn luyện Perceptron hoàn tất.")


Huấn luyện Perceptron hoàn tất.


In [9]:
y_pred = model.predict(X_test)
net_scores = model.decision_function(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=[CLASS_NAMES[0], CLASS_NAMES[1]],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))


Accuracy: 0.8102

Classification Report:
                    precision    recall  f1-score   support

         Phổ thông       0.75      0.72      0.73        50
Giá/mật độ giá cao       0.84      0.86      0.85        87

          accuracy                           0.81       137
         macro avg       0.80      0.79      0.79       137
      weighted avg       0.81      0.81      0.81       137

Confusion Matrix:
[[36 14]
 [12 75]]


In [10]:
perceptron = model.named_steps["perceptron"]

print("Bias b:", round(float(perceptron.intercept_[0]), 4))
print("Số vòng lặp:", perceptron.n_iter_)
print("5 giá trị net đầu tiên:", np.round(net_scores[:5], 4))


Bias b: 0.2
Số vòng lặp: 10
5 giá trị net đầu tiên: [0.5274 0.98   1.7298 0.834  1.3832]


In [11]:
# Kết quả trên tập test
test_result = df.loc[X_test.index].copy()
test_result["nhan_that"] = y_test
test_result["du_doan"] = y_pred
test_result["net"] = net_scores
test_result["ket_qua"] = np.where(
    test_result["nhan_that"] == test_result["du_doan"],
    "Đúng", "Sai"
)

display(test_result.head(20))
print("Số mẫu dự đoán sai:", (test_result["ket_qua"] == "Sai").sum())


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,price_per_area,segment,phan_khuc,nhan_that,du_doan,net,ket_qua
39,7910000,6000,4,2,4,yes,no,no,no,yes,1,no,semi-furnished,1318.333333,1,Giá/mật độ giá cao,1,1,0.527414,Đúng
200,4900000,4520,3,1,2,yes,no,yes,no,yes,0,no,semi-furnished,1084.070796,1,Giá/mật độ giá cao,1,1,0.979995,Đúng
23,8645000,4560,3,2,2,yes,yes,yes,no,yes,1,no,furnished,1895.833333,1,Giá/mật độ giá cao,1,1,1.729760,Đúng
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished,736.312500,0,Phổ thông,0,1,0.834007,Sai
179,5215000,3180,3,2,2,yes,no,no,no,no,2,no,semi-furnished,1639.937107,1,Giá/mật độ giá cao,1,1,1.383246,Đúng
214,4865000,4350,2,1,1,yes,no,yes,no,no,0,no,unfurnished,1118.390805,1,Giá/mật độ giá cao,1,1,0.039900,Đúng
431,3290000,3180,4,1,2,yes,no,yes,no,yes,0,no,unfurnished,1034.591195,1,Giá/mật độ giá cao,1,1,1.206976,Đúng
34,8120000,6840,5,1,2,yes,yes,yes,no,yes,1,no,furnished,1187.134503,1,Giá/mật độ giá cao,1,0,-0.152345,Sai
463,3080000,3090,2,1,1,yes,yes,yes,no,no,0,no,unfurnished,996.763754,1,Giá/mật độ giá cao,1,1,1.198228,Đúng
150,5600000,5136,3,1,2,yes,yes,yes,no,yes,0,yes,unfurnished,1090.342679,1,Giá/mật độ giá cao,1,1,1.609257,Đúng


Số mẫu dự đoán sai: 26


In [12]:
# Xuất toàn bộ dữ liệu đã phân khúc
output_file = BASE_DIR / "phan_khuc_nha.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

# Xuất kết quả test
test_output = BASE_DIR / "ket_qua_test_phan_khuc_nha.csv"
test_result.to_csv(test_output, index=False, encoding="utf-8-sig")

print("Đã tạo:", output_file)
print("Đã tạo:", test_output)


Đã tạo: d:\AI\Documents\Downloads\lac-hong-neural-network\phan_khuc_nha.csv
Đã tạo: d:\AI\Documents\Downloads\lac-hong-neural-network\ket_qua_test_phan_khuc_nha.csv


## Dự đoán một căn nhà mới


## Kết luận

- Dữ liệu lấy từ `Housing.csv`.
- `price_per_area` dùng để tạo nhãn.
- `price` và `price_per_area` không dùng làm feature để tránh **data leakage**.
- Perceptron được huấn luyện sau khi chuẩn hóa dữ liệu số và One-Hot Encoding dữ liệu phân loại.
- Kết quả được đánh giá bằng Accuracy, Classification Report và Confusion Matrix.
- `phan_khuc_nha.csv` chứa dữ liệu cùng thông tin phân khúc.
